# Multi-Query Demo：LlamaIndex

使用 `QueryFusionRetriever` 生成多个查询，并融合不同查询的检索结果。

In [ ]:
# 如果环境中还没有 LlamaIndex，可先执行：
# %pip install llama-index llama-index-llms-google-genai llama-index-embeddings-huggingface

import os
from dotenv import load_dotenv
from llama_index.core import Settings, SimpleDirectoryReader, VectorStoreIndex
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.google_genai import GoogleGenAI

load_dotenv()
Settings.llm = GoogleGenAI(
    model="gemini-3.1-flash-lite",
    api_key=os.environ["GEMINI_API_KEY"],
    temperature=0,
)
Settings.embed_model = HuggingFaceEmbedding(model_name="moka-ai/m3e-base")

In [ ]:
documents = SimpleDirectoryReader(
    input_dir="../knowledge_db/prompt_engineering",
    recursive=True,
).load_data()
index = VectorStoreIndex.from_documents(documents)
base_retriever = index.as_retriever(similarity_top_k=4)
retriever = QueryFusionRetriever(
    [base_retriever],
    llm=Settings.llm,
    num_queries=4,
    mode="reciprocal_rerank",
    use_async=False,
    verbose=True,
)

In [ ]:
question = "总结文本转换这篇文章的主要观点、方法和示例"
nodes = retriever.retrieve(question)
context = "\n\n".join(node.node.get_content() for node in nodes)
response = Settings.llm.complete(
    f"只根据以下上下文回答，并覆盖多个片段，不要只总结一个示例。\n\n上下文：{context}\n问题：{question}"
)
print(response)

`from llama_index.core import Settings`


```python
from llama_index.core import Settings
```

`Settings` 是 LlamaIndex 的全局配置对象，用来统一设置默认的 LLM、Embedding 模型等组件。

例如：

```python
from llama_index.core import Settings
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.llm = GoogleGenAI(
    model="gemini-3.1-flash-lite",
    api_key=os.environ["GEMINI_API_KEY"],
)

Settings.embed_model = HuggingFaceEmbedding(
    model_name="moka-ai/m3e-base"
)
```

之后创建索引时：

```python
index = VectorStoreIndex.from_documents(documents)
```

LlamaIndex 会自动使用：

```python
Settings.llm
Settings.embed_model
```

## 如果不使用 Settings

也可以每次单独传入：

```python
index = VectorStoreIndex.from_documents(
    documents,
    embed_model=embedding_model, # 重复传入
)

query_engine = index.as_query_engine(
    llm=llm, # 重复传入
)
```

但这样每个组件都要显式传递。

## 简单理解

```text
Settings = LlamaIndex 的默认配置中心
```

类似于：

```python
DEFAULT_LLM = ...
DEFAULT_EMBEDDING = ...
```

设置一次：

```python
Settings.llm = llm
Settings.embed_model = embedding_model
```

后续很多 LlamaIndex 组件会自动使用这些配置。
